# Code Critic — Project Description

**Course**: Watspeed Agentic AI Capstone  
**Author**: Jayaprakash Sivanandam  
**Date**: 2026-04-15  

---

## About This Project

This document describes the **Code Critic** agentic AI application at a project level.
It is organized into the following sections:

| Section | Topic |
|---------|-------|
| 1 | Project Concept |
| 2 | Framework & Harness |
| 3 | Prompting Techniques |
| 4 | Orchestration |
| 5 | Memory |
| 6 | Tool Use |
| 7 | Safeguards / Guardrails |


## 1. Project Concept

**Code Critic** is an agentic AI application designed to automate SQL code review for Snowflake data warehouses.
Given the name of a stored procedure or an ad-hoc SQL query, the application connects to Snowflake using
**Snowpark Python**, retrieves the full DDL or query text, and then routes it through a graph of specialized
AI agents — built with **LangGraph** and powered by **Google Gemini** — that independently analyze the code
for performance inefficiencies, security vulnerabilities, and style violations.
The agents' findings are merged into a single ranked `FindingsReport` that includes a severity rating,
a plain-language description of each issue, the original offending snippet, and a rewritten version
of the code with a line-by-line explanation of every change.
The goal is to give data engineers and analytics engineers the same depth of review they would get from
a senior peer, delivered automatically and consistently, without leaving their existing Snowflake workflow.

| Component | Choice | Reason |
|-----------|--------|--------|
| Agent orchestration | LangGraph | Provides a StateGraph with typed state, conditional routing, and a Send API for parallel fan-out — exactly what multi-agent coordination requires. |
| LLM | Google Gemini (langchain-google-genai) | Strong code understanding, native JSON mode for structured output, and tight LangChain/LangGraph integration. |
| Snowflake integration | Snowpark Python (snowflake-snowpark-python) | First-party Snowflake library that bundles the connector and a DataFrame API; supports RSA role pinning at session creation. |
| Development environment | Jupyter Notebooks (.ipynb) | Mandated by course guidelines; each notebook begins with a description and readme section. |
| Structured output | Pydantic FindingsReport model | Validates every agent's output before it passes to the next node, catching malformed LLM responses early. |
| Dependency management | venv + requirements.txt | Isolated environment; key packages: langgraph, langchain-google-genai, snowflake-snowpark-python, pydantic, python-dotenv. |
| Secret management | keyring + configparser | Password retrieved from Windows Credential Manager at runtime via keyring; non-sensitive connection parameters loaded from config.ini using configparser. No python-dotenv required. |

## 3. Prompting Techniques

Each LangGraph node will use a dedicated system prompt scoped to that node's
responsibility. Rather than one large catch-all prompt, each agent will receive focused
instructions that define its role, what to analyze, and the expected output format.
The prompting approach will be refined through testing as the application is built.

The following techniques are planned:

- **Persona framing** — each analyzer node will open with a role that anchors its
  reasoning to a specific engineering domain. For example, the `router` node might use
  a lightweight classifier persona, while the `performance_analyzer` node would use
  a persona oriented around query optimization and warehouse efficiency.

- **Chain-of-thought** — analyzer nodes will include a step-by-step reasoning instruction
  before producing findings, to encourage more thorough analysis on complex queries.

- **Structured output** — all nodes will be instructed to respond in the `FindingsReport`
  JSON schema, paired with Gemini's native JSON mode to enforce this at the API level.

- **Scope restriction** — each node's prompt will explicitly state what to ignore, not
  just what to analyze, to keep agents from overlapping with each other's domain.

- **Context injection** — the `synthesizer` node will receive prior analysis history
  as part of its prompt to help identify regressions or improvements across runs.

| Node | Persona Intent | Techniques Planned |
|---|---|---|
| router | Lightweight classifier — determine input type only | Structured output, scope restriction |
| schema_fetcher | Metadata retrieval agent — fetch DDL and execution plan | ReAct (reason then tool call), scope restriction |
| performance_analyzer | Query optimization and warehouse efficiency focus | Persona framing, chain-of-thought, structured output |
| security_auditor | Security and access control focus | Persona framing, chain-of-thought, structured output |
| style_reviewer | Code readability and naming convention focus | Persona framing, chain-of-thought, structured output |
| synthesizer | Merge and rank findings from all three analyzers | Context injection, structured output, prompt chaining |

### Prompt Management — Skills Folder

System prompts and persona definitions will be managed as standalone text files in a
dedicated `skills/` folder, separate from application code. Each node will have its own
prompt file that can be edited and tested independently without touching the graph logic.
This keeps prompt iteration lightweight — a prompt can be updated, versioned, and tested
without modifying any Python files.

The planned project structure is as follows:

```
code_critic/
│
├── config.ini                        
├── requirements.txt
├── .gitignore
│
├── notebooks/
│   └── code_critic.ipynb             
│
├── skills/                           
│   ├── router.txt
│   ├── schema_fetcher.txt
│   ├── performance_analyzer.txt
│   ├── security_auditor.txt
│   ├── style_reviewer.txt
│   └── synthesizer.txt
│
├── src/
│   ├── graph.py                      
│   ├── nodes.py                      
│   ├── tools.py                      
│   ├── models.py                     
│   ├── state.py                      
│   └── check_secrets.py              
│
└── memory/
    └── analysis_history.json      

```   

At runtime, each node will load its persona and instructions from the corresponding
file in `skills/` before assembling the final prompt:

```python
# intended pattern — exact implementation to be finalized during development
def load_skill(node_name: str) -> str:
    with open(f"skills/{node_name}.txt", "r") as f:
        return f.read()

# inside each node
system_prompt = load_skill("performance_analyzer")
```

This approach keeps the graph code stable while allowing prompt content to evolve
independently. It also makes it straightforward to swap or compare different persona
definitions during testing without changing any node logic.

The exact prompt text for each node will be finalized during development and testing.

## 4. Orchestration

The application uses a **LangGraph `StateGraph`** with five specialized agent nodes connected by typed edges.
A shared `AgentState` TypedDict flows through the graph, accumulating results at each step.

```
[User Input: object name or SQL text]
          │
          ▼
       router
       │     │
  (SP path) (query path)
       │     │
       ▼     ▼
   schema_fetcher          ← Snowpark tools called here
          │
          ▼  (parallel fan-out via LangGraph Send API)
 ┌────────┬──────────┬───────────┐
 │        │          │           │
 ▼        ▼          ▼           │
perf_   security_  style_        │
analyzer auditor  reviewer       │
 │        │          │           │
 └────────┴──────────┘           │
          │                      │
          ▼                      │
      synthesizer ◄──────────────┘
          │
          ▼
    [FindingsReport]
```

**Key orchestration decisions:**

- **Conditional routing** in `router`: a lightweight Gemini call classifies the input and sets a flag in `AgentState`; LangGraph conditional edges direct the graph to the stored-procedure DDL fetch path or the ad-hoc query path accordingly.
- **Parallel fan-out**: The `Send` API dispatches `performance_analyzer`, `security_auditor`, and `style_reviewer` concurrently. Each writes its partial `FindingsReport` into `AgentState`; the `synthesizer` waits for all three before merging.
- **Isolation**: Each analyzer node receives only the DDL text and its own system prompt — it cannot read another analyzer's output, preventing anchoring bias between agents.

## 5. Memory

The application uses a **two-tier memory strategy** to support both interactive follow-up questions
within a session and persistent knowledge across sessions.

---

### Short-Term Memory — LangGraph `MemorySaver`

LangGraph's built-in `MemorySaver` checkpointer serializes the full `AgentState` to an in-memory
store after every graph step, keyed by a `thread_id` (e.g., `SCHEMA.SP_NAME_20260415T103000`).

**What this enables:**
- A user can ask follow-up questions in the same session (*"Why did you flag that join?"*,
  *"Show me only the security findings"*) without re-fetching DDL from Snowflake.
- If the graph is interrupted mid-run (e.g., a Gemini timeout), the checkpointer allows the run
  to resume from the last completed node rather than restarting from scratch.

**Scope**: In-memory only; cleared when the Python process exits. Not persisted to disk.

---

### Long-Term Memory — Local JSON File

After each completed analysis, the final `FindingsReport` is appended to a local file at
`memory/analysis_history.json`, keyed by `"SCHEMA.OBJECT_NAME"`.

```json
{
  "ANALYTICS.SP_DAILY_REVENUE": [
    {
      "analyzed_at": "2026-04-10T09:15:00",
      "findings_summary": ["Missing clustering key on DATE_ID", "Dynamic SQL in line 42"]
    }
  ]
}
```

**What this enables:**
- When the same object is analyzed again, the `schema_fetcher` node loads the prior entry and
  injects it into the `synthesizer` prompt as historical context. The synthesizer can then
  highlight *regressions* (old issues that reappeared) and *improvements* (old issues now fixed).
- No external database is required; plain Python `json` module reads and writes the file.

**Scope**: Persists across sessions. The file is excluded from version control (`.gitignore`).

## 6. Tool Use

Two LangGraph tools are defined and bound exclusively to the `schema_fetcher` node.
Both use a **Snowpark `Session`** initialized at application startup with a read-only role.

---

### Tool 1 — `snowpark_fetch_ddl`

Retrieves the full source DDL of a stored procedure or the text of a recent ad-hoc query.

```python
# Stored procedure DDL
session.sql("SELECT GET_DDL('procedure', ?)", [object_name]).collect()

# Ad-hoc query text from query history
session.sql("""
    SELECT QUERY_TEXT FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY())
    WHERE QUERY_ID = ?
""", [query_id]).collect()
```

### Tool 2 — `snowpark_explain_plan`

Fetches the logical execution plan for a SQL statement, giving the `performance_analyzer`
concrete cost and operator data to reason about.

```python
session.sql(f"EXPLAIN USING JSON {validated_sql}").collect()
```

(`validated_sql` has already passed the object-name regex guard before reaching this tool.)

---

**Both tools are strictly read-only** (SELECT and EXPLAIN only). The Snowflake role assigned
to the session (`CODE_CRITIC_RO`) enforces this at the database level as a second layer of
protection, independent of application logic.

## 7. Safeguards / Guardrails

Five independent layers protect the application against misuse, data leakage, and unsafe outputs.

---

## Layer 1 — Read-Only Snowflake Role (Database-Level)

A dedicated Snowflake role `CODE_CRITIC_RO` is created with the minimum privileges required:

```sql
CREATE ROLE CODE_CRITIC_RO;
GRANT USAGE ON WAREHOUSE CRITIC_WH TO ROLE CODE_CRITIC_RO;
GRANT USAGE ON DATABASE ANALYTICS_DB TO ROLE CODE_CRITIC_RO;
GRANT USAGE ON SCHEMA ANALYTICS_DB.PUBLIC TO ROLE CODE_CRITIC_RO;
GRANT SELECT ON ALL TABLES IN SCHEMA ANALYTICS_DB.INFORMATION_SCHEMA TO ROLE CODE_CRITIC_RO;
```

**Authentication method: Windows Credential Manager + config.ini.**
Non-sensitive connection parameters (account, user, warehouse, database, role) are stored
in a `config.ini` file that is committed to version control. The Snowflake password is stored
separately as a Generic Credential in Windows Credential Manager, encrypted by the OS
using the logged-in user's account, and retrieved at runtime via the `keyring` library.
No plaintext secrets ever appear in source code or version control.

**`config.ini`** (committed to version control):
```ini
[snowflake]
account   = your_account
user      = your_username
role      = CODE_CRITIC_RO
warehouse = your_warehouse
database  = your_database
```

**One-time credential setup** (run once interactively — never commit with real values):
```python
import keyring

keyring.set_password("snowflake", "password", "your_password")
```

After setup, the password appears under **Windows Credentials → Generic Credentials**
in the Windows Credential Manager UI (`control /name Microsoft.CredentialManager`).

The Snowpark session is built by combining both sources at runtime:

```python
import keyring
import configparser
from snowflake.snowpark import Session

config = configparser.ConfigParser()
config.read("config.ini")

sf = config["snowflake"]

connection_params = {
    "account":   sf["account"],
    "user":      sf["user"],
    "role":      sf["role"],      # pinned — cannot be elevated
    "warehouse": sf["warehouse"],
    "database":  sf["database"],
    "password":  keyring.get_password("snowflake", "password"),  # from Windows Credential Manager
}

session = Session.builder.configs(connection_params).create()
```

Even if an agent accidentally generates a DML statement, Snowflake rejects it with
"Insufficient privileges" — no application-level check required.

---

### Layer 2 — SQL Injection Prevention (Input Guard)

All user-supplied object names are validated against a strict allowlist regex before
any Snowpark call is made:

```python
import re
SAFE_IDENTIFIER = re.compile(r'^[A-Za-z0-9_.]+$')
if not SAFE_IDENTIFIER.match(object_name):
    raise ValueError(f"Invalid object name: {object_name!r}")
```

All parameterized values (query IDs, object names) are passed via Snowpark's `?` binding —
never string-interpolated into SQL.

---

### Layer 3 — DML / DDL Output Blocker (Output Guard)

After the `synthesizer` node produces its `FindingsReport`, a post-processing step scans
every `suggested_fix` field for modification keywords:

```python
BLOCKED_KEYWORDS = {"INSERT", "UPDATE", "DELETE", "CREATE", "ALTER", "DROP", "TRUNCATE", "MERGE"}
```

Any finding whose suggestion contains one of these keywords is flagged `severity = BLOCKED`
and its `suggested_fix` is replaced with a plain-English description of the change
(*"Replace the dynamic INSERT with a static SELECT INTO..."*). Agents suggest; they never
emit executable modification code.

---

### Layer 4 — Pydantic Output Validation (Schema Guard)

Every agent node's raw LLM response is parsed through the `FindingsReport` Pydantic model
before it is written to `AgentState`:

```python
try:
    report = FindingsReport.model_validate_json(raw_response)
except ValidationError:
    # retry the node call (up to 2 retries), then surface an error to the user
```

This prevents malformed LLM output (missing fields, wrong types) from silently propagating
through the graph and corrupting the final report.

---

### Layer 5 — Secret Detection Hook (Credential Guard)

A reusable utility module (`check_secrets.py`) is called before every LLM submission
across all nodes. Rather than checking the output after the fact, the guard runs against
the fully assembled prompt before it is sent to Gemini — this is the higher-risk moment,
as DDL retrieved from Snowflake could contain hardcoded credentials left in stored
procedure code by a developer.

The check will scan for patterns such as hardcoded passwords, API keys, bearer tokens,
private key blocks, and long base64-like strings. If a match is found, the LLM call is
blocked and an error is surfaced to the user. If the prompt is clean, execution continues
normally.

The intended call pattern inside each node is:

```python
# 1. Build the prompt from AgentState
prompt = <assembled from DDL, execution plan, or findings as appropriate>

# 2. Check for secrets before sending to LLM
check_for_secrets(prompt, context="<node_name>")

# 3. If no secrets detected, proceed with LLM call
response = llm.invoke(...)
```

This pattern will be applied consistently across all nodes that assemble and submit
prompts: `router`, `schema_fetcher`, `performance_analyzer`, `security_auditor`,
`style_reviewer`, and `synthesizer`. The `context` parameter identifies which node
triggered the block, making issues easier to trace during development.

The exact patterns used for detection will be finalized during development and testing.